# Policy Agent — Team Walkthrough

This notebook walks your team through **building, deploying, and testing** the policy agent end-to-end:

1. Environment check (gcloud, ADC, Python)
2. Clone repo and install dependencies
3. Configure project / region / buckets
4. Create GCS buckets and seed sample policies
5. Deploy to Vertex AI Agent Engine with **Agent Identity** (runs `deploy.py`)
6. Capture Reasoning Engine resource name
7. Register as a Gemini Enterprise add-on agent (runs `register_gemini_enterprise.py`)
8. Test the agent in Gemini Enterprise — **expected to fail** (missing permissions)
9. Grant the agent principal `roles/storage.objectViewer` on the policy bucket
10. Re-test with new permissions — should succeed
11. Logging & monitoring (audit logs, metrics, alerts)
12. Teardown (stop billing when you're done)



## 1. Environment check

Verify the prerequisites before you do anything that costs money.

**Common gotcha:** the `gcloud` CLI active account and your Application Default Credentials (ADC) can be *different* accounts. The Python SDK uses ADC, so if they differ you'll see `403 storage.buckets.get` during deploy. The cell below prints both — they should match (or at least both have access to the target project).

In [ ]:
import shutil, subprocess, sys

print('Python :', sys.version.split()[0])
for tool in ('gcloud', 'gsutil'):
    path = shutil.which(tool)
    print(f'{tool:7}: {path or "NOT FOUND \u2014 install the Google Cloud SDK"}')


def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as e:
        return f'ERROR: {e.output.strip()}'


print('\ngcloud active account :', sh('gcloud config get-value account'))
print('gcloud active project :', sh('gcloud config get-value project'))

# ADC identity = what the Python SDK / deploy.py will use.  If this differs
# from the gcloud active account you'll see 403s during deploy.
print('\nADC identity (what the Python SDK uses):')
print(sh(
    'curl -s -H "Authorization: Bearer '
    '$(gcloud auth application-default print-access-token)" '
    'https://openidconnect.googleapis.com/v1/userinfo'
))


If ADC is missing or wrong, run this in a terminal (opens a browser), then restart the kernel:
```
gcloud auth application-default login --account=<your-account>
```

## 2. Clone repo and install dependencies

Installs the package in editable mode so code edits in `policy_agent/` are picked up without reinstalling.

> **Note:** If you are running this notebook from **inside** the already-cloned repo , the clone step is automatically skipped and the cell just does a `git pull` + `pip install`. No action needed.

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/avnit/agent-identity.git'

# Pick a sensible default per environment:
#   - Colab            \u2192 /content/agent-identity
#   - Local Jupyter    \u2192 the parent of notebooks/, i.e. the repo root
if os.path.isdir('/content'):
    DEFAULT_REPO_DIR = '/content/agent-identity'
else:
    DEFAULT_REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

REPO_DIR = DEFAULT_REPO_DIR  # @param {type:"string"}
os.environ['REPO_DIR'] = REPO_DIR
print(f'Repo root: {REPO_DIR}')

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already present \u2014 pulling latest...')
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr or '(already up to date)')
else:
    print(f'Cloning {REPO_URL} \u2192 {REPO_DIR} ...')
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed:\n{r.stderr}')
    print(r.stdout or 'Clone complete')

print('\nInstalling package (editable)...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
print('\u2705 Install complete')


## 3. Configuration

**Edit this cell** — everything below reads from these variables. These match the env vars expected by `deploy.py` and `register_gemini_enterprise.py`.

| Variable | Required by | Notes |
|---|---|---|
| `PROJECT_ID` | deploy.py | Your GCP project ID |
| `LOCATION` | deploy.py | Vertex AI region (e.g. `us-central1`) |
| `STAGING_BUCKET` | deploy.py | Agent Engine staging bucket (must be globally unique) |
| `POLICY_BUCKET` | deploy.py | Bucket holding policy documents (globally unique) |
| `POLICY_PREFIX` | deploy.py | Object prefix for policies (default `policies/`) |
| `DISPLAY_NAME` | deploy.py | Name shown in Agent Engine console |
| `AGENT_MODEL` | policy_agent | Gemini model to use |
| `GEMINI_ENTERPRISE_APP_ID` | register_gemini_enterprise.py | Your GE app ID |
| `GEMINI_ENTERPRISE_LOCATION` | register_gemini_enterprise.py | `global`, `us`, or `eu` |

In [ ]:
import os

# \u2500\u2500 REQUIRED \u2014 edit these \u2500\u2500
PROJECT_ID                 = 'your-project-id'   # @param {type:"string"}
LOCATION                   = 'us-central1'       # @param {type:"string"}
STAGING_BUCKET             = ''                  # @param {type:"string"}
POLICY_BUCKET              = ''                  # @param {type:"string"}
POLICY_PREFIX              = 'policies/'         # @param {type:"string"}
DISPLAY_NAME               = 'policy-agent'      # @param {type:"string"}
AGENT_MODEL                = 'gemini-2.5-flash'  # @param {type:"string"}
GEMINI_ENTERPRISE_APP_ID   = ''                  # @param {type:"string"}
GEMINI_ENTERPRISE_LOCATION = 'global'            # @param {type:"string"}

# Filled in Step 6 (after deploy.py prints the resource name)
REASONING_ENGINE = ''

# \u2500\u2500 Auto-derive bucket names if left blank \u2500\u2500
if not STAGING_BUCKET:
    STAGING_BUCKET = f'{PROJECT_ID}-agent-staging'
if not POLICY_BUCKET:
    POLICY_BUCKET = f'{PROJECT_ID}-policies'

# \u2500\u2500 Validate \u2500\u2500
assert PROJECT_ID and PROJECT_ID != 'your-project-id', \
    '\u274c Set PROJECT_ID to your actual GCP project ID before proceeding'

# \u2500\u2500 Propagate to subprocess env (deploy.py / register_gemini_enterprise.py) \u2500\u2500
env_map = {
    'GOOGLE_CLOUD_PROJECT':        PROJECT_ID,
    'GOOGLE_CLOUD_LOCATION':       LOCATION,
    'STAGING_BUCKET':              STAGING_BUCKET,
    'POLICY_BUCKET':               POLICY_BUCKET,
    'POLICY_PREFIX':               POLICY_PREFIX,
    'DISPLAY_NAME':                DISPLAY_NAME,
    'AGENT_MODEL':                 AGENT_MODEL,
    'GEMINI_ENTERPRISE_APP_ID':    GEMINI_ENTERPRISE_APP_ID,
    'GEMINI_ENTERPRISE_LOCATION':  GEMINI_ENTERPRISE_LOCATION,
}
for k, v in env_map.items():
    os.environ[k] = v

print('Configuration:')
for k in env_map:
    print(f'  {k:30} = {env_map[k]}')
print(f'  {"REASONING_ENGINE":30} = {REASONING_ENGINE or "(set in Step 6)"}')
print('\n\u2705 Configuration set')


## 4. Enable APIs, create buckets, seed sample policies

Idempotent — re-runs no-op if the API is already enabled or the bucket already exists.

In [ ]:
import subprocess

def run(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0 and 'already exists' not in (r.stderr or '').lower():
        print('STDERR:', r.stderr.strip())
    return r

run(f'gcloud services enable aiplatform.googleapis.com storage.googleapis.com discoveryengine.googleapis.com --project={PROJECT_ID}')
run(f'gcloud storage buckets create gs://{STAGING_BUCKET} --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage buckets create gs://{POLICY_BUCKET}  --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage cp {REPO_DIR}/sample_policies/*.md gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')
run(f'gcloud storage ls gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')

## 5. Deploy to Vertex AI Agent Engine with Agent Identity

**This step creates billable resources** — ~5–15 minutes while the container builds.

This runs `deploy.py` from the repo root. The key config in that script is `identity_type=AGENT_IDENTITY`, which makes Agent Engine mint a per-agent SPIFFE principal you can grant IAM to directly.

The script prints the `reasoningEngine` resource name at the end — it's captured automatically into `REASONING_ENGINE` below.

In [ ]:
import subprocess, sys, os, re

deploy_script = os.path.join(REPO_DIR, 'deploy.py')

print(f'Running: python {deploy_script}')
print('This takes 5\u201315 minutes \u2014 streaming stdout/stderr below...\n')

# Stream output line-by-line so the build progress is actually visible
proc = subprocess.Popen(
    [sys.executable, '-u', deploy_script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=os.environ.copy(),
)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line)
returncode = proc.wait()

if returncode != 0:
    raise RuntimeError(f'deploy.py exited with code {returncode}')

# Try to auto-extract the reasoningEngine resource name from the captured output
m = re.search(r'projects/[^/\s]+/locations/[^/\s]+/reasoningEngines/\d+', ''.join(captured))
if m:
    REASONING_ENGINE = m.group(0)
    os.environ['REASONING_ENGINE'] = REASONING_ENGINE
    print(f'\n\u2705 Auto-captured REASONING_ENGINE = {REASONING_ENGINE}')
else:
    print('\n\u26a0\ufe0f  Could not auto-extract reasoningEngine \u2014 paste it into Step 6.')

print('\n\u2705 Deployment complete')


## 6. Capture Reasoning Engine resource name ( Manual step )

Once deploy.py has finished executing, navigate to the Reasoning Engine section of the Google Cloud Console to locate your newly deployed agent.

### Step A: Copy Agent Metadata
Identify the specific agent you just created and copy its Identity and Resource Name. This unique identifier is essential for the subsequent registration, IAM permission granting, and remote testing phases.

### Step B: Configure Telemetry & Logging
To ensure full visibility into your agent's performance, modify the Telemetry collection settings:

Instrumentation: Enable OpenTelemetry traces and logs.

Prompt & Response Logging: Enable the logging of all prompt inputs and generated outputs.

Storage Destinations: Select both Cloud Logging and Cloud Storage (GCS) as the destinations for these logs.

> **Tip:** The value looks like:
> `projects/123456789/locations/us-central1/reasoningEngines/987654321098765432`

In [ ]:
# Paste the reasoningEngine printed by deploy.py (if Cell 11 didn't auto-capture it).
# Leave blank to keep whatever was auto-captured.
REASONING_ENGINE_OVERRIDE = ''  # @param {type:"string"}

if REASONING_ENGINE_OVERRIDE:
    REASONING_ENGINE = REASONING_ENGINE_OVERRIDE

assert REASONING_ENGINE, '\u274c Paste the value printed by deploy.py into REASONING_ENGINE_OVERRIDE'
os.environ['REASONING_ENGINE'] = REASONING_ENGINE
print(f'REASONING_ENGINE = {REASONING_ENGINE}')

# Sanity-check Gemini Enterprise vars too \u2014 these are needed in Step 7
assert GEMINI_ENTERPRISE_APP_ID, \
    '\u274c Set GEMINI_ENTERPRISE_APP_ID in Step 3 before proceeding to Step 7'
print(f'GEMINI_ENTERPRISE_APP_ID   = {GEMINI_ENTERPRISE_APP_ID}')
print(f'GEMINI_ENTERPRISE_LOCATION = {GEMINI_ENTERPRISE_LOCATION}')


## 7. Register as a Gemini Enterprise add-on agent

Runs `register_gemini_enterprise.py` from the repo root. This POSTs to the Discovery Engine Assistants API to register the deployed reasoning engine as an add-on agent in your Gemini Enterprise app.

**Prerequisites:**
- `GEMINI_ENTERPRISE_APP_ID` must be set in Step 3
- `REASONING_ENGINE` must be set from Step 5
- A Gemini Enterprise app must already exist in this project

In [ ]:
import subprocess, sys, os

assert GEMINI_ENTERPRISE_APP_ID, \
    '❌ Set GEMINI_ENTERPRISE_APP_ID in Step 3 before running this cell'
assert REASONING_ENGINE, \
    '❌ Set REASONING_ENGINE in Step 6 before running this cell'

# REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
register_script = os.path.join(REPO_DIR, 'register_gemini_enterprise.py')

print(f'Running: python {register_script}')
print(f'  GEMINI_ENTERPRISE_APP_ID = {GEMINI_ENTERPRISE_APP_ID}')
print(f'  REASONING_ENGINE         = {REASONING_ENGINE}\n')

result = subprocess.run(
    [sys.executable, register_script],
    capture_output=True,
    text=True,
    cwd=REPO_DIR,
    env=os.environ.copy(),
)

if result.returncode != 0:
    raise RuntimeError(f'register_gemini_enterprise.py exited with code {result.returncode}')

print('\n✅ Agent registered with Gemini Enterprise')

## 8. Test in Gemini Enterprise — ⚠️ Expected to fail

Go to the **Gemini Enterprise** app in the Google Cloud console and ask the Policy Assistant agent a question, e.g.:

> *"What is my work from home policy?"*

**This will fail** with a permission error — the agent principal has not yet been granted access to the policy bucket. You will see an error like:

```
403 storage.objects.list denied on gs://<POLICY_BUCKET>
```

This is expected and demonstrates why Agent Identity + IAM binding matters. Continue to Step 8 to fix it.

---

**Where to find the app:**
1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Navigate to **Gemini Enterprise** (or Agentspace)
3. Open your app and select the **Policy Assistant** agent
4. Send a test question and observe the error

## 9. Grant the agent principal access to GCS

Fetches the reasoning engine's `effectiveIdentity` (the per-agent SPIFFE principal) and binds `roles/storage.objectViewer` on the policy bucket to it.

This is the fix for the failure in Step 7 — no service-account key needed.

In [ ]:
import json, subprocess

assert REASONING_ENGINE, '\u274c Set REASONING_ENGINE in Step 6 before running this cell'

token = subprocess.check_output(
    'gcloud auth application-default print-access-token', shell=True, text=True
).strip()

resp = subprocess.check_output(
    f'curl -s -H "Authorization: Bearer {token}" '
    f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}',
    shell=True, text=True,
)
engine = json.loads(resp)

# effectiveIdentity has lived in different places across API versions.  Try
# common paths and fall back to dumping the response so you can see where it is.
def _pluck(d, *paths):
    for path in paths:
        cur = d
        for key in path:
            if not isinstance(cur, dict) or key not in cur:
                cur = None
                break
            cur = cur[key]
        if cur:
            return cur
    return None

effective_identity = _pluck(
    engine,
    ('spec', 'effectiveIdentity'),
    ('effectiveIdentity',),
    ('spec', 'agentIdentity', 'effectiveIdentity'),
)

if not effective_identity:
    print('Raw engine response:')
    print(json.dumps(engine, indent=2)[:2000])
    raise RuntimeError(
        'Could not locate effectiveIdentity. Inspect the response above and '
        'update the path. Confirm Agent Identity was enabled in deploy.py '
        '(identity_type=AGENT_IDENTITY).'
    )

# effectiveIdentity arrives as a SPIFFE path (no scheme), e.g.
#   agents.global.org-123.system.id.goog/resources/aiplatform/projects/...
# IAM principal format is:  principal://<that path>
# Strip a leading 'spiffe://' if the API ever returns it.
if effective_identity.startswith('spiffe://'):
    effective_identity = effective_identity[len('spiffe://'):]
AGENT_PRINCIPAL = f'principal://{effective_identity}'
os.environ['AGENT_PRINCIPAL'] = AGENT_PRINCIPAL
print(f'Agent principal: {AGENT_PRINCIPAL}\n')

# Bind roles/storage.objectViewer on the policy bucket
run(
    f'gcloud storage buckets add-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)
print('\n\u2705 IAM binding applied \u2014 allow ~30 seconds for propagation')


## 10. Re-test in Gemini Enterprise — ✅ Should succeed

Now that the agent principal has `roles/storage.objectViewer` on the policy bucket, go back to the Gemini Enterprise app and ask the same question again:

> *"What is my work from home policy?"*

The agent should now successfully:
1. Call `list_policies` → `search_policies` → `get_policy_document`
2. Read from `gs://<POLICY_BUCKET>/policies/` via its **Agent Identity** principal (no service-account key)
3. Return a cited answer from the policy documents

You can also run a quick remote test from the notebook to confirm:

In [ ]:
import vertexai

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=dict(api_version='v1beta1'),
)

remote = client.agent_engines.get(name=REASONING_ENGINE)


async def ask_remote(prompt: str, session_id: str | None = None) -> tuple[str, str]:
    if session_id is None:
        session = await remote.async_create_session(user_id='nb-user')
        session_id = session['id']
    out = []
    async for event in remote.async_stream_query(
        user_id='nb-user', session_id=session_id, message=prompt,
    ):
        for p in (event.get('content') or {}).get('parts', []):
            if 'text' in p:
                out.append(p['text'])
    return ''.join(out), session_id


print('Q1: What is my work from home policy?\n')
answer1, sid = await ask_remote('What is my work from home policy?')
print(answer1)

print('\n---\nQ2 (same session, follow-up):\n')
answer2, _ = await ask_remote('Which policies cover travel and expenses?', session_id=sid)
print(answer2)


## 11. Logging & Monitoring

Now that the agent is working end-to-end, wire up observability so you can answer four questions in production:

| Question | Where to look |
|---|---|
| Is the agent serving requests reliably? | Cloud Monitoring built-in metrics on `aiplatform.googleapis.com/ReasoningEngine` |
| What did the agent *do* on a given request? | Cloud Trace (OpenTelemetry) + agent stdout in Cloud Logging |
| Did the **agent principal** (not a human) read the policy bucket? | Cloud Storage Data Access audit logs, filtered by SPIFFE `principalSubject` |
| When did the failures in Step 8 happen, and did they stop after the IAM binding? | Audit logs filtered on `severity=ERROR` |

The cells below set this up. Most of it is one-time configuration per project.

> **Heads-up on cost:** Data Access audit logs for Cloud Storage are off by default *because they're high-volume*. For a read-only policy bucket the volume is trivial, but if you reuse this pattern on a busy bucket, scope the config carefully.


### 11.a Enable Data Access audit logs for Cloud Storage

This is the cell that makes the Agent Identity story *provable*. Without DATA_READ logs on `storage.googleapis.com`, you can't show an auditor that it was the agent's SPIFFE principal — not a service-account key, not a human — that read the policy documents.

We also enable DATA_READ on `aiplatform.googleapis.com` so you capture every `streamQuery`/session call against the reasoning engine.


In [ ]:
import json, subprocess, tempfile, os

# Read current project IAM policy
policy_json = subprocess.check_output(
    f'gcloud projects get-iam-policy {PROJECT_ID} --format=json',
    shell=True, text=True,
)
policy = json.loads(policy_json)

# Merge in audit configs for the two services we care about
desired = {
    'storage.googleapis.com':    ['ADMIN_READ', 'DATA_READ', 'DATA_WRITE'],
    'aiplatform.googleapis.com': ['ADMIN_READ', 'DATA_READ', 'DATA_WRITE'],
}
existing = {a['service']: a for a in policy.get('auditConfigs', [])}
for service, log_types in desired.items():
    cfg = existing.get(service, {'service': service, 'auditLogConfigs': []})
    have = {c['logType'] for c in cfg['auditLogConfigs']}
    for lt in log_types:
        if lt not in have:
            cfg['auditLogConfigs'].append({'logType': lt})
    existing[service] = cfg
policy['auditConfigs'] = list(existing.values())

with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
    json.dump(policy, f)
    policy_path = f.name

run(f'gcloud projects set-iam-policy {PROJECT_ID} {policy_path} --quiet')
os.unlink(policy_path)
print('\u2705 Data Access audit logs enabled on storage + aiplatform')


### 11.b Query the agent's logs and audit trail

Three log queries that should be in your muscle memory for this stack:

1. **Agent stdout / model + tool errors** — emitted by the reasoning engine itself.
2. **Step 8 IAM denials** — the *expected* failures before you bound the role. Useful to confirm they stopped after Step 9.
3. **Agent principal reading the policy bucket** — the proof that Agent Identity is doing its job (no key, no impersonation).


In [ ]:
import subprocess, shlex, os

REASONING_ENGINE_ID = REASONING_ENGINE.rsplit('/', 1)[-1]
AGENT_PRINCIPAL = os.environ.get('AGENT_PRINCIPAL', '')  # set by Cell 18


def logs(filter_str, limit=20, freshness='1h'):
    cmd = (
        f'gcloud logging read {shlex.quote(filter_str)} '
        f'--project={PROJECT_ID} --limit={limit} --freshness={freshness} '
        f'--format="value(timestamp,severity,protoPayload.methodName,'
        f'protoPayload.authenticationInfo.principalSubject,'
        f'protoPayload.status.message,textPayload)"'
    )
    return subprocess.check_output(cmd, shell=True, text=True)


# 1. Agent runtime logs (stdout from the reasoning engine container)
print('=' * 72)
print('1) Reasoning engine runtime logs (last 1h)')
print('=' * 72)
print(logs(
    f'resource.type="aiplatform.googleapis.com/ReasoningEngine" '
    f'AND resource.labels.reasoning_engine_id="{REASONING_ENGINE_ID}"'
))

# 2. IAM-denied storage access (the Step 8 failure mode)
print('=' * 72)
print('2) Storage access DENIED on the policy bucket (last 24h)')
print('=' * 72)
print(logs(
    f'protoPayload.serviceName="storage.googleapis.com" '
    f'AND protoPayload.resourceName:"projects/_/buckets/{POLICY_BUCKET}" '
    f'AND severity=ERROR',
    freshness='24h',
))

# 3. Successful reads BY THE AGENT PRINCIPAL \u2014 the Agent Identity proof
if AGENT_PRINCIPAL:
    spiffe_id = 'spiffe://' + AGENT_PRINCIPAL.removeprefix('principal://')
    print('=' * 72)
    print('3) Policy bucket reads attributed to the agent SPIFFE principal')
    print('   ', spiffe_id)
    print('=' * 72)
    print(logs(
        f'protoPayload.serviceName="storage.googleapis.com" '
        f'AND protoPayload.resourceName:"projects/_/buckets/{POLICY_BUCKET}" '
        f'AND protoPayload.authenticationInfo.principalSubject="{spiffe_id}"',
        freshness='24h',
    ))
else:
    print('(Skip step 3 \u2014 run Step 9 first to populate AGENT_PRINCIPAL)')


### 11.c Built-in metrics + a p99 latency alert

The Agent Engine exposes a standard set of metrics on the `aiplatform.googleapis.com/ReasoningEngine` monitored resource: request count, request latencies, error rate, model token usage, and per-tool call counts/latencies. The cell below pulls the last 1 hour of `request_count` (broken out by `response_code`) and `request_latencies` p99 directly from the Monitoring API, then creates a threshold alert that fires when p99 latency exceeds 30s for 5 minutes.

The default Cloud Console dashboard is **Vertex AI Agent Engine Overview** — open it directly with the deeplink printed at the bottom.


In [ ]:
import json, subprocess, tempfile, os, time

# pip install -q google-cloud-monitoring  # uncomment if not already installed
from google.cloud import monitoring_v3

client = monitoring_v3.MetricServiceClient()
project_name = f'projects/{PROJECT_ID}'

now = int(time.time())
interval = monitoring_v3.TimeInterval({
    'end_time':   {'seconds': now},
    'start_time': {'seconds': now - 3600},
})

filter_base = (
    f'resource.type="aiplatform.googleapis.com/ReasoningEngine" '
    f'AND resource.labels.reasoning_engine_id="{REASONING_ENGINE_ID}"'
)

print('Request count (last 1h, by response_code):')
for ts in client.list_time_series(request={
    'name': project_name,
    'filter': f'metric.type="aiplatform.googleapis.com/reasoning_engine/request_count" AND {filter_base}',
    'interval': interval,
    'view': monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
}):
    code_label = ts.metric.labels.get('response_code', '?')
    total = sum(p.value.int64_value for p in ts.points)
    print(f'  response_code={code_label:>3}  total={total}')

print('\nRequest latency p99 (last 1h, ms):')
for ts in client.list_time_series(request={
    'name': project_name,
    'filter': f'metric.type="aiplatform.googleapis.com/reasoning_engine/request_latencies" AND {filter_base}',
    'interval': interval,
    'view': monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
}):
    points = [p.value.distribution_value for p in ts.points if p.value.distribution_value]
    if points:
        # Approximate p99 from the most recent distribution
        d = points[0]
        if d.bucket_options.exponential_buckets and d.bucket_counts:
            cumulative = 0
            target = 0.99 * d.count
            growth = d.bucket_options.exponential_buckets.growth_factor
            scale  = d.bucket_options.exponential_buckets.scale
            for i, cnt in enumerate(d.bucket_counts):
                cumulative += cnt
                if cumulative >= target:
                    print(f'  ~p99 \u2248 {scale * (growth ** i):.0f} ms (mean={d.mean:.0f} ms, n={int(d.count)})')
                    break

# \u2500\u2500 Create / update an alert policy: p99 latency > 30s for 5min \u2500\u2500
alert_policy = {
    'displayName': f'PolicyAgent p99 latency >30s ({REASONING_ENGINE_ID})',
    'combiner': 'OR',
    'conditions': [{
        'displayName': 'p99 request_latencies > 30000ms',
        'conditionThreshold': {
            'filter': (
                'metric.type="aiplatform.googleapis.com/reasoning_engine/request_latencies" '
                f'AND resource.labels.reasoning_engine_id="{REASONING_ENGINE_ID}"'
            ),
            'aggregations': [{
                'alignmentPeriod': '60s',
                'perSeriesAligner': 'ALIGN_PERCENTILE_99',
                'crossSeriesReducer': 'REDUCE_MAX',
            }],
            'comparison': 'COMPARISON_GT',
            'thresholdValue': 30000,
            'duration': '300s',
        },
    }],
    'enabled': True,
}

with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
    json.dump(alert_policy, f)
    policy_path = f.name

print('\nCreating alert policy...')
out = subprocess.run(
    f'gcloud alpha monitoring policies create --policy-from-file={policy_path} '
    f'--project={PROJECT_ID} --format="value(name)"',
    shell=True, capture_output=True, text=True,
)
os.unlink(policy_path)
print(out.stdout.strip() or out.stderr.strip())

# Console deeplink to the default dashboard
print(
    f'\n\u2192 Default dashboard: '
    f'https://console.cloud.google.com/vertex-ai/agents/locations/{LOCATION}'
    f'/agent-engines/{REASONING_ENGINE_ID}/metrics?project={PROJECT_ID}'
)


### 11.d Cloud Trace, dashboards, and where this fits in a VPC-SC perimeter

A few pointers rather than more code:

- **Cloud Trace** — Agent Engine ships OpenTelemetry traces by default; each `streamQuery` becomes a parent span with child spans per tool call. Open `https://console.cloud.google.com/traces/list?project=<PROJECT_ID>` and filter by `service.name="reasoning-engine"` to see per-request spans (model latency, `list_policies` / `search_policies` / `get_policy_document` tool latency, GCS round-trip).
- **Per-agent dashboard** — In the console go to **Vertex AI → Agent Engine → \<your agent\> → Overview / Models / Tools / Usage**. The Tools tab is the highest-signal view for this notebook because it surfaces `tool_error_rate` per tool ID — if the bucket binding regresses, the `get_policy_document` tool will spike here before users complain.
- **VPC Service Controls** — if this project is inside a perimeter, the Reasoning Engine Service Agent (`service-<PROJECT_NUMBER>@gcp-sa-aiplatform-re.iam.gserviceaccount.com`) needs an **ingress rule** allowing it to reach `storage.googleapis.com` and `artifactregistry.googleapis.com`. Without it, deploy succeeds but every tool call fails inside the perimeter. The agent SPIFFE principal also has VPC-SC support (Preview) for ingress/egress rules — useful when you want the agent itself (not just the service agent) on the allow-list.
- **SCC Enterprise** — Agent Engine Threat Detection (Preview) is part of Security Command Center; turn it on at the org or project level to surface prompt-injection patterns, anomalous tool sequences, and credential-exfiltration attempts against deployed agents.
- **BigQuery sink (optional)** — for retention beyond Cloud Logging's default 30 days, sink the audit + agent logs to BigQuery:
  ```bash
  gcloud logging sinks create policy-agent-sink \
    bigquery.googleapis.com/projects/$PROJECT_ID/datasets/agent_audit \
    --log-filter='resource.type="aiplatform.googleapis.com/ReasoningEngine" OR (protoPayload.serviceName="storage.googleapis.com" AND protoPayload.resourceName:"buckets/'"$POLICY_BUCKET"'")' \
    --project=$PROJECT_ID
  ```


## 12. Teardown (stop ongoing billing)

Deletes the reasoning engine. The buckets stay (empty buckets cost ~nothing) — the stale IAM binding on the policy bucket becomes a no-op once the principal is gone, but you can remove it if you like.

In [ ]:
import subprocess
token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
print(subprocess.check_output(
    f'curl -s -X DELETE -H "Authorization: Bearer {token}" '
    f'"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}?force=true"',
    shell=True, text=True,
))

In [ ]:
# Optional: remove the now-stale IAM binding
run(
    f'gcloud storage buckets remove-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)

## Troubleshooting

| Error you see | Fix |
|---|---|
| `403 storage.buckets.get` during deploy | ADC identity ≠ gcloud identity. Re-run `gcloud auth application-default login --account=<right-account>` and restart the kernel. |
| `400 Environment variable name 'GOOGLE_CLOUD_PROJECT' is reserved` | Don't put `GOOGLE_CLOUD_PROJECT` / `GOOGLE_CLOUD_LOCATION` in `env_vars` — Agent Engine injects them. |
| `ModuleNotFoundError: No module named 'policy_agent'` in engine logs | `extra_packages=['policy_agent']` must be set so the local source ships to the container. |
| `403 storage.objects.list denied` at query time | Step 8 didn't run or was run against the wrong bucket. Re-check `POLICY_BUCKET` and the agent principal. |
| `Missing required env var: STAGING_BUCKET` | Make sure Step 3 ran and `os.environ` was populated before running Step 5. |
| Deploy succeeds but engine won't start | `gcloud logging read 'resource.type="aiplatform.googleapis.com/ReasoningEngine" AND resource.labels.reasoning_engine_id="<ID>"' --project=<PROJECT> --limit=50 --freshness=30m` |